# Team MLB · SVM Extra Credit (TrafficSliver 분할 트래픽)

노션 **Extra Credit > 모델팀 실험 규칙**을 따른다. 평가는 모두 **같은 extra split test**, 학습 데이터만 바꾼다.

| 조건 | 학습 | 보려는 것 |
| --- | --- | --- |
| A | 일반 train (메인 trim0 특징) | 분할 방어가 기존 모델을 얼마나 막는가 |
| B | split train | 분할 트래픽으로 학습하면 얼마나 회복하는가 |
| C | 일반 train + split train | 섞으면 B보다 나은가 |

| 항목 | 설정 |
| --- | --- |
| 데이터 버전 | `include`(빈 split = 0 벡터 포함) / `exclude`(빈 split 행 제외) |
| 과제 | Closed-world(필수, 95 클래스) / Open-world 이진(선택, mon 1 / unmon −1) |
| 특징 | 공통 110개, SVM 추가 조건 g1+g6 53개 (Closed만) |
| 하이퍼파라미터 | 메인 best 고정 — Closed 110: C=2⁷, γ=2⁻⁶ / Closed 53: C=2¹⁴, γ=2⁻² / 이진 110: C=2³, γ=2⁻⁴ |
| 전처리 | 메인과 같음: g1~g5 `log1p` → 상수 특징 제거 → 표준화 → RBF SVC. 조건별 학습 데이터에서만 fit |
| 분할 | `splits/extra_split_seed42.csv`를 `source_group_id`로 merge. 같은 방문의 split 5행은 같은 쪽 |
| 확률 | `CalibratedClassifierCV(sigmoid, ensemble=False)`에 분할 파일의 **`fold`(방문 단위)**를 cv로 넘긴다. `probability=True`의 내부 Platt 보정은 행 단위 5-fold라 규칙의 "행 단위 KFold 금지"와 맞지 않아 쓰지 않는다 |
| 주 지표 | 방문 단위 정확도·macro-F1: 방문의 split 확률 평균 → argmax (제외 버전은 남은 split만 평균) |
| 보조 지표 | 행 단위 정확도, 빈 행을 오답으로 센 행 단위 정확도 |

- **제외 버전**은 데이터팀의 `*_without_zero.csv.gz`를 따로 받지 않고, 포함 버전 CSV에서 `status == "empty"` 행을 걸러 만든다. 남는 행 수 172,085는 `validation_without_zero.json` 보고값과 같다.
- 실행 환경: 로컬 PC (Colab 아님). 결과는 조건·버전마다 끝나는 대로 저장하고, 이미 있으면 건너뛴다.

## 0. 설정

In [ ]:
import os, time
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "splits").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
FEATURE_MAIN = ROOT / "features" / "기본 실험"      # 메인 trim0 특징 (저장소)
EXTRA_DIR    = ROOT / "data" / "extra"               # extra 특징 (저장소에 올리지 않음)
SPLIT_MAIN   = ROOT / "splits" / "split_seed42.csv"
SPLIT_EXTRA  = ROOT / "splits" / "extra_split_seed42.csv"
RESULT_DIR   = ROOT / "results" / "svm" / "extra_credit"

SEED, N_FOLDS = 42, 5
N_JOBS   = int(os.environ.get("EC_N_JOBS", 5))   # 보정용 fold 병렬 수 (메모리에 맞춰 조절)
CACHE_MB = 300                                    # SVC 커널 캐시
SKIP_IF_EXISTS = True

# 메인 best (log2C, log2gamma) — svm_exp4_summary.csv, svm_summary.csv(closed_world, g1+g6)
BEST = {("closed", "110"): (7, -6), ("closed", "g1g6_53"): (14, -2), ("binary", "110"): (3, -4)}

CLOSED_TRAIN_FRAC = float(os.environ.get("EC_CLOSED_FRAC", "0.3")) # Closed B·C 학습 방문 샘플링 비율 (아래 4장 참고)
RUN_BINARY        = os.environ.get("EC_RUN_BINARY", "0") == "1"   # Open-world 이진(선택)
BINARY_TRAIN_FRAC = float(os.environ.get("EC_BINARY_FRAC", "0.3")) # 이진 B·C 학습 방문 샘플링 비율

USE_GPU = os.environ.get("EC_GPU") == "1"   # 1이면 RAPIDS cuML의 cuml.accel로 sklearn SVC를 GPU에서 실행 (Linux + NVIDIA GPU)

SMOKE = os.environ.get("EC_SMOKE") == "1"          # 코드 점검용: 방문 3%만 사용, 결과는 별도 폴더
if SMOKE:
    RESULT_DIR = Path(os.environ["EC_SMOKE_DIR"])
RESULT_DIR.mkdir(parents=True, exist_ok=True)
print("ROOT:", ROOT); print("RESULT_DIR:", RESULT_DIR)
print(f"USE_GPU={USE_GPU}, N_JOBS={N_JOBS}, CLOSED_TRAIN_FRAC={CLOSED_TRAIN_FRAC}, RUN_BINARY={RUN_BINARY}, BINARY_TRAIN_FRAC={BINARY_TRAIN_FRAC}, SMOKE={SMOKE}")

In [ ]:
if USE_GPU:   # sklearn을 import하기 전에 켜야 한다
    get_ipython().run_line_magic("load_ext", "cuml.accel")
    N_JOBS = 1   # GPU 하나를 여러 프로세스가 나눠 쓰지 않도록 fold를 순서대로 학습
    print("cuml.accel 활성화, N_JOBS=1")

import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.feature_selection import VarianceThreshold
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score, confusion_matrix, roc_auc_score, average_precision_score
import sklearn
print("sklearn", sklearn.__version__)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)

## 1. extra split 특징 로드 · 분할 merge · 점검

In [ ]:
META = ["record_id", "source_group_id", "variant", "status", "dataset", "page_id", "sample_idx", "label", "trim"]
GPREF = tuple(f"g{i}_" for i in range(1, 7))

ex = pd.read_csv(EXTRA_DIR / "features_extra_split_trim0.csv.gz")
FEATS_110 = [c for c in ex.columns if c.startswith(GPREF)]
assert list(ex.columns[:9]) == META and len(FEATS_110) == 110 and (ex.trim == 0).all()

sp = pd.read_csv(SPLIT_EXTRA)
assert sp.source_group_id.is_unique and len(sp) == 38000
ex = ex.merge(sp[["source_group_id", "label", "split", "fold", "source"]].rename(columns={"label": "label_sp"}),
              on="source_group_id", how="left", validate="many_to_one")
assert ex.split.notna().all(), "분할 파일에 없는 방문"
assert (ex.label == ex.label_sp).all(), "라벨 불일치"
assert (ex.groupby("source_group_id").size() == 5).all(), "방문당 split 5행이 아님"
assert ex.groupby("source_group_id").split.nunique().max() == 1
assert set(sp.source_group_id) == set(ex.source_group_id)

ex = ex.copy()
ex["is_empty"] = ex.status.eq("empty")
Xc = ex[FEATS_110].to_numpy(float)
assert np.isfinite(Xc).all()
assert ((Xc == 0).all(axis=1) == ex.is_empty.to_numpy()).all(), "empty 표시와 0 벡터가 다름"
assert not ex.groupby("source_group_id").is_empty.all().any(), "split 5개가 모두 빈 방문이 있음"
assert (~ex.is_empty).sum() == 172085, "제외 버전 행 수가 validation_without_zero.json과 다름"
del Xc

mon_te = ex[(ex.dataset == "mon") & (ex.split == "test")].drop_duplicates("source_group_id")
assert (mon_te.label.value_counts() == 40).all() and mon_te.label.nunique() == 95
print(pd.crosstab([ex.dataset, ex.split], ex.status, margins=True))
print("\n방문 수:", ex.drop_duplicates("source_group_id").groupby("split").size().to_dict())

## 2. 메인 특징 로드 · 방문 대응 (조건 A·C)

- 대응 규칙: extra mon `(site p, sample s)` = 메인 `(page p×10 + s//20, sample s%20)`, extra unmon `i` = 메인 `unmon10k_i`
- 메인 train 기록은 extra에서도 모두 train이어야 한다 → A·C 학습에 extra test 방문이 들어가지 않는다.
- 조건 C의 확률 보정 fold는 메인 기록에도 같은 방문의 extra `fold`를 준다.

In [ ]:
mm = pd.read_csv(FEATURE_MAIN / "features_mon_trim0.csv")
mu = pd.read_csv(FEATURE_MAIN / "features_unmon10k_trim0.csv")
main = pd.concat([mm, mu], ignore_index=True)
assert [c for c in main.columns if c.startswith(GPREF)] == FEATS_110, "메인/extra 특징 열이 다름"

msp = pd.read_csv(SPLIT_MAIN)
main["split"] = main.record_id.map(dict(zip(msp.record_id, msp.split)))
assert main.split.notna().all()

is_mon = main.dataset.eq("mon")
gid = pd.Series(index=main.index, dtype=object)
gid[is_mon] = ("mon_" + (main.page_id[is_mon] // 10).astype(str) + "-"
               + ((main.page_id[is_mon] % 10) * 20 + main.sample_idx[is_mon]).astype(str))
gid[~is_mon] = "unmon_" + main.record_id[~is_mon].str.split("_").str[1].astype(int).astype(str)
main["source_group_id"] = gid.astype(str)

sp_idx = sp.set_index("source_group_id")
main = main.join(sp_idx[["label", "split", "fold"]].add_suffix("_ex"), on="source_group_id")
assert main.split_ex.notna().all()
assert (main.label == main.label_ex).all(), "메인-extra 라벨 불일치"
tr = main.split.eq("train")
assert (main.loc[tr, "split_ex"] == "train").all(), "메인 train 방문이 extra test에 있음 (누수)"
print("메인 test인데 extra train인 방문:", int(((main.split == "test") & (main.split_ex == "train")).sum()),
      "(extra test 평가에는 영향 없음)")
main["fold"] = main.fold_ex.astype(int)
main["variant"], main["status"], main["is_empty"] = "main", "ok", False
print(pd.crosstab(main.dataset, main.split))

In [ ]:
if SMOKE:   # 코드 점검용 축소: 라벨별 방문 3%
    keep = set(sp.groupby("label").sample(frac=0.03, random_state=SEED).source_group_id)
    ex, main = ex[ex.source_group_id.isin(keep)].reset_index(drop=True), main[main.source_group_id.isin(keep)].reset_index(drop=True)
    print("SMOKE:", len(ex), "extra rows,", len(main), "main rows")

## 3. 공통 함수

In [ ]:
FEATSETS = {"110": FEATS_110, "g1g6_53": [c for c in FEATS_110 if c.startswith(("g1_", "g6_"))]}
assert len(FEATSETS["g1g6_53"]) == 53
COLS = ["source_group_id", "record_id", "variant", "dataset", "label", "fold", "is_empty"]

def safe_log1p(X):
    return np.log1p(np.clip(X, 0, None))

def make_svm(feats, log2C, log2g):
    log_cols = [c for c in feats if not c.startswith("g6_")]
    pre = ColumnTransformer([("log", FunctionTransformer(safe_log1p, feature_names_out="one-to-one"), log_cols)],
                            remainder="passthrough", verbose_feature_names_out=False)
    return Pipeline([("pre", pre), ("var", VarianceThreshold()), ("sc", StandardScaler()),
                     ("svc", SVC(kernel="rbf", C=2.0 ** log2C, gamma=2.0 ** log2g, cache_size=CACHE_MB))])

def task_view(df, task, feats):
    d = df[df.dataset == "mon"] if task == "closed" else df
    d = d[COLS + feats].copy()
    d["y"] = d.label.astype(int) if task == "closed" else np.where(d.dataset == "mon", 1, -1)
    return d

def train_frame(task, cond, version, feats, frac=None):
    parts = []
    if cond in ("A", "C"):
        parts.append(task_view(main[main.split == "train"], task, feats))
    if cond in ("B", "C"):
        e = ex[ex.split == "train"]
        if version == "exclude":
            e = e[~e.is_empty]
        parts.append(task_view(e, task, feats))
    d = pd.concat(parts, ignore_index=True)
    if frac is not None and frac < 1:   # 방문 단위 샘플링 (방문의 split 5개·메인 기록이 함께), 라벨 층화
        v = d.drop_duplicates("source_group_id")
        keep = set(v.groupby("label").sample(frac=frac, random_state=SEED).source_group_id)
        d = d[d.source_group_id.isin(keep)].reset_index(drop=True)
    return d

def test_frame(task, version, feats):
    e = ex[ex.split == "test"]
    if version == "exclude":
        e = e[~e.is_empty]
    return task_view(e, task, feats).reset_index(drop=True)

def fit_model(d, feats, hp):
    folds = d.fold.to_numpy().astype(int)
    assert set(folds) == set(range(N_FOLDS))
    cv = [(np.flatnonzero(folds != k), np.flatnonzero(folds == k)) for k in range(N_FOLDS)]
    model = CalibratedClassifierCV(make_svm(feats, *hp), method="sigmoid", cv=cv, ensemble=False, n_jobs=N_JOBS)
    t0 = time.time(); model.fit(d[feats], d.y.to_numpy()); fit_sec = time.time() - t0
    n_sv = int(model.calibrated_classifiers_[0].estimator[-1].n_support_.sum())
    return model, fit_sec, n_sv

In [ ]:
def visit_scores(te, P, classes):
    pv = pd.DataFrame(P, columns=classes)
    pv["gid"] = te.source_group_id.to_numpy()
    vm = pv.groupby("gid").mean()
    vy = te.groupby("source_group_id").y.first().loc[vm.index].astype(int)
    return vm, vy

def evaluate(te, P, classes, task, n_visits):
    y = te.y.to_numpy()
    correct = classes[P.argmax(1)] == y
    vm, vy = visit_scores(te, P, classes)
    assert len(vm) == n_visits, "test 방문 수가 다름"
    vpred = np.asarray(classes)[vm[list(classes)].to_numpy().argmax(1)]
    out = dict(visit_acc=float((vpred == vy.to_numpy()).mean()),
               visit_macro_f1=float(f1_score(vy, vpred, average="macro")),
               row_acc=float(correct.mean()),
               row_acc_empty_wrong=float((correct & ~te.is_empty.to_numpy()).sum() / (5 * n_visits)))
    if task == "binary":
        s = vm[1].to_numpy(); yb = (vy.to_numpy() == 1)
        tn, fp, fn, tp = confusion_matrix(yb, vpred == 1, labels=[False, True]).ravel()
        out.update(visit_tpr=tp / (tp + fn), visit_fpr=fp / (fp + tn), visit_precision=tp / (tp + fp) if tp + fp else 0.0,
                   visit_roc_auc=roc_auc_score(yb, s), visit_pr_auc=average_precision_score(yb, s),
                   visit_majority_baseline=max(yb.mean(), 1 - yb.mean()))
    return out

KEY = ["model", "task", "condition", "version", "features"]
SUMMARY_COLS = KEY + ["n_train_rows", "n_test_rows", "n_test_visits", "visit_acc", "visit_macro_f1",
                      "row_acc", "row_acc_empty_wrong", "fit_sec", "note"]
SUMMARY, DETAILS = RESULT_DIR / "ec_summary.csv", RESULT_DIR / "svm_ec_details.csv"

def upsert(path, row):
    df = pd.read_csv(path) if path.exists() else pd.DataFrame(columns=list(row))
    if len(df):
        same = np.ones(len(df), bool)
        for k in KEY:
            same &= df[k].astype(str).to_numpy() == str(row[k])
        df = df[~same]
    pd.concat([df, pd.DataFrame([row])], ignore_index=True).to_csv(path, index=False)

def proba_path(task, cond, version, fkey):
    return RESULT_DIR / f"ec_test_proba_{task}_{cond}_{version}{'' if fkey == '110' else '_' + fkey}.csv"

def is_done(task, cond, version, fkey):
    # 요약 행만 있으면 끝난 것으로 본다. 확률 파일은 커서 저장소에 올리지 않으므로 다른 PC에서 이어 돌릴 때 없을 수 있다.
    if not (SKIP_IF_EXISTS and SUMMARY.exists()):
        return False
    s = pd.read_csv(SUMMARY, dtype=str)
    return ((s.task == task) & (s.condition == cond) & (s.version == version) & (s.features == fkey)).any()

def save_proba(path, te, P, classes):
    out = te[["record_id", "source_group_id", "variant", "y"]].reset_index(drop=True)
    out = pd.concat([out, pd.DataFrame(P, columns=[f"p_{c}" for c in classes])], axis=1)
    out.sort_values("record_id").to_csv(path, index=False, float_format="%.6f")

In [ ]:
MODELS = {}   # 조건 A 모델 (join 참고 평가에 재사용)

def run(task, cond, fkey, versions=("include", "exclude"), frac=None):
    todo = [v for v in versions if not is_done(task, cond, v, fkey)]
    if not todo:
        print(f"[skip] {task} {cond} {fkey} {versions}: 결과 있음"); return
    feats, hp = FEATSETS[fkey], BEST[(task, fkey)]
    n_visits = test_frame(task, "include", feats).source_group_id.nunique()
    fitted = {}
    for v in todo:
        tv = "include" if cond == "A" else v          # A는 메인으로 한 번만 학습
        if tv not in fitted:
            d = train_frame(task, cond, tv, feats, frac=frac if cond != "A" else None)
            print(f"[fit] {task} {cond} {fkey} train={tv}: {len(d):,} rows / {d.source_group_id.nunique():,} visits ...", flush=True)
            fitted[tv] = (*fit_model(d, feats, hp), len(d), d.source_group_id.nunique())
            print(f"      fit {fitted[tv][1] / 60:.1f} min, support vectors {fitted[tv][2]:,}", flush=True)
        model, fit_sec, n_sv, n_tr, n_tr_v = fitted[tv]
        if cond == "A":
            MODELS[(task, fkey)] = model
        te = test_frame(task, v, feats)
        t0 = time.time(); P = model.predict_proba(te[feats]); pred_sec = time.time() - t0
        m = evaluate(te, P, model.classes_, task, n_visits)
        notes = ["확률: CalibratedClassifierCV(sigmoid, ensemble=False, cv=extra_split fold)"]
        if v == "exclude":
            notes.append("제외 버전은 포함 CSV에서 status=empty 행을 걸러 만듦")
        if fkey != "110":
            notes.append(f"SVM 추가 특징 g1+g6 53개, 메인 best C=2^{hp[0]} gamma=2^{hp[1]}")
        if frac is not None and frac < 1 and cond != "A":
            notes.append(f"학습 방문 {frac:.0%} 샘플링 (라벨 층화, 방문 단위, seed {SEED})")
        if USE_GPU:
            notes.append("GPU: RAPIDS cuml.accel (sklearn SVC 대체 구현)")
        if SMOKE:
            notes.append("SMOKE")
        row = dict(model="SVM", task=task, condition=cond, version=v, features=fkey,
                   n_train_rows=n_tr, n_test_rows=len(te), n_test_visits=n_visits,
                   **{k: m[k] for k in ["visit_acc", "visit_macro_f1", "row_acc", "row_acc_empty_wrong"]},
                   fit_sec=round(fit_sec, 1), note="; ".join(notes))
        upsert(SUMMARY, row)
        upsert(DETAILS, {**{k: row[k] for k in KEY}, "log2C": hp[0], "log2gamma": hp[1], "n_support": n_sv,
                         "n_train_visits": n_tr_v, "train_frac": frac if cond != "A" else None,
                         "predict_sec": round(pred_sec, 1), **{k: v_ for k, v_ in m.items() if k.startswith("visit_") and k not in row}})
        save_proba(proba_path(task, cond, v, fkey), te, P, model.classes_)
        print(f"  → {v}: visit_acc {m['visit_acc']:.4f} | visit_F1 {m['visit_macro_f1']:.4f} | row_acc {m['row_acc']:.4f} "
              f"| row_acc_empty_wrong {m['row_acc_empty_wrong']:.4f} | test rows {len(te):,}", flush=True)

## 4. Closed-world

규칙 5장대로 **조건 B(포함 버전)를 먼저** 돌려 시간을 쟀다.

- 전체 학습 방문(76,000행)으로 B 포함을 돌린 결과 **학습 65분 + 예측 16분 = 81분**이었다(1시간 초과). 학습 행 76,000개 중 75,684개가 support vector였다.
- 같은 방식이면 Closed 전체가 10시간 이상 걸리므로, 규칙 5장대로 **B·C는 학습 방문 30%만 샘플링**한다(라벨 층화, 방문의 split 5개와 대응 메인 기록을 함께, seed 42). A는 메인 train 전체를 쓴다.
- 전체 데이터로 돌린 B 포함 결과는 `condition = B_full100` 참고 행으로 남겨, 30% 샘플링이 결과를 얼마나 바꾸는지 직접 비교한다.

In [ ]:
T0 = time.time()
run("closed", "B", "110", versions=("include",), frac=CLOSED_TRAIN_FRAC)
print(f"조건 B(포함) 소요: {(time.time() - T0) / 60:.1f} min")

In [ ]:
for cond in ["A", "B", "C"]:
    run("closed", cond, "110", frac=CLOSED_TRAIN_FRAC)
for cond in ["A", "B", "C"]:
    run("closed", cond, "g1g6_53", frac=CLOSED_TRAIN_FRAC)

## 5. Open-world 이진 (선택)

- 특징 110개만. extra는 mon:unmon = 1:1이라 메인 이진 수치(19,000:10,000)와 직접 비교하지 않는다.
- B·C 학습 행이 15만 개가 넘어 RBF SVM 학습 시간이 크게 늘어난다. 규칙 5장대로 **학습 방문 일부만 샘플링**한다(방문의 split 5개와 대응 메인 기록을 함께, 라벨 층화). 조건 A(메인 23,200행)는 전체를 쓴다.

In [ ]:
if RUN_BINARY:
    for cond in ["A", "B", "C"]:
        run("binary", cond, "110", frac=BINARY_TRAIN_FRAC)
else:
    print("RUN_BINARY=False → 이진 과제 건너뜀")

## 6. (선택) 조건 A 모델로 join test 참고 평가

`join`은 분할 경로를 합친 별도 관찰이다(방문당 1행). 기본 조건에는 넣지 않고, 메인으로 학습한 모델이 합친 관찰에서 얼마나 맞히는지만 참고로 본다.

In [ ]:
JOIN_PATH = EXTRA_DIR / "features_extra_join_trim0.csv.gz"
if JOIN_PATH.exists():
    jn = pd.read_csv(JOIN_PATH).merge(sp[["source_group_id", "split", "fold"]], on="source_group_id", how="left", validate="one_to_one")
    assert jn.split.notna().all() and (jn.variant == "join").all()
    jn["is_empty"] = jn.status.eq("empty")
    jn = jn[jn.split == "test"]
    rows = []
    for task, fkey in [("closed", "110"), ("closed", "g1g6_53")] + ([("binary", "110")] if RUN_BINARY else []):
        feats = FEATSETS[fkey]
        if (task, fkey) not in MODELS:
            d = train_frame(task, "A", "include", feats)
            MODELS[(task, fkey)] = fit_model(d, feats, BEST[(task, fkey)])[0]
        te = task_view(jn, task, feats)
        P = MODELS[(task, fkey)].predict_proba(te[feats]); pred = MODELS[(task, fkey)].classes_[P.argmax(1)]
        rows.append(dict(model="SVM", task=task, condition="A", eval="join_test", features=fkey, n_test_rows=len(te),
                         acc=float((pred == te.y.to_numpy()).mean()), macro_f1=float(f1_score(te.y, pred, average="macro"))))
    jref = pd.DataFrame(rows); jref.to_csv(RESULT_DIR / "svm_ec_join_reference.csv", index=False)
    print(jref.round(4).to_string(index=False))
else:
    print("join 파일 없음 → 건너뜀")

## 7. 결과 요약

In [ ]:
s = pd.read_csv(SUMMARY)
print(s.drop(columns=["note", "model"]).round(4).to_string(index=False))
print()
print(pd.read_csv(DETAILS).drop(columns=["model"]).round(4).to_string(index=False))
print("\n메인 Closed-world test 정확도 (비교 기준): SVM 110개 0.8521 / SVM g1+g6 0.8650")